# 11 — kvpress: LongBench Benchmark with PrefillDecodingPress

This notebook evaluates KV cache compression on two
[LongBench](https://github.com/THUDM/LongBench) tasks using
[kvpress](https://github.com/NVIDIA/kvpress) with Qwen3-8B:

- **gov_report** — government report summarization (~8.7K words). Generates
  one-page summaries, providing **high decoding stress**.
- **hotpotqa** — multi-document QA (~9.2K words). Tests multi-hop reasoning
  across scattered paragraphs.

We test **KeyDiffPress**-based compression applied to both **prefill** and
**decoding** phases using PrefillDecodingPress, with two decoding strategies:
- **full_replacement** — CompressionRatioDecodingPress
- **filtering** — FilteringPress

Scoring uses the HuggingFace `evaluate` library:
- gov_report: ROUGE-L (via `evaluate.load("rouge")`)
- hotpotqa: F1 (via `evaluate.load("squad")`)

Results are saved to `results/kvpress_longbench/` for comparison in later notebooks.

## Configuration

In [1]:
import os
os.environ["HF_HOME"] = "/opt/app-root/src/.cache/huggingface"

MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

FRACTION = 0.01

LONGBENCH_TASKS = ["gov_report", "hotpotqa"]

MAX_NEW_TOKENS = {
    "gov_report": 512,
    "hotpotqa": 64,
}

PRESS_CONFIGS = {
    "full_replacement": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=CompressionRatioDecodingPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
        ),
    ),
    "filtering": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=FilteringPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
            fill_padding=False,
        ),
    ),
}

In [2]:
import sys
import builtins

_original_print = builtins.print

def print(*args, **kwargs):
    _original_print(*args, **kwargs)
    if sys.stdout is not sys.__stdout__:
        kwargs['file'] = sys.__stdout__
        kwargs['flush'] = True
        _original_print(*args, **kwargs)

In [3]:
import sys
import os

FORK_DIR = "/opt/app-root/src/kvpress-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import kvpress
    print(f"Using FORK kvpress from {FORK_DIR}")
else:
    import kvpress
    print(f"Using SYSTEM kvpress")

print(f"  location: {os.path.dirname(kvpress.__file__)}")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using FORK kvpress from /opt/app-root/src/kvpress-fork
  location: /opt/app-root/src/kvpress-fork/kvpress
Using FORK kvpress from /opt/app-root/src/kvpress-fork
  location: /opt/app-root/src/kvpress-fork/kvpress


## 1. Load Model

In [4]:
import torch
from transformers import pipeline
from kvpress import (
    KeyDiffPress, PrefillDecodingPress, CompressionRatioDecodingPress,
    FilteringPress,
)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} — {vram_gb:.1f} GB VRAM")

model_kwargs = {}

try:
    import flash_attn  # noqa: F401
    model_kwargs["attn_implementation"] = "flash_attention_2"
    print("Using Flash Attention 2")
except ImportError:
    print("Flash Attention 2 not available, using default attention")

pipe = pipeline(
    "kv-press-text-generation",
    model=MODEL_NAME,
    device_map="auto",
    model_kwargs=model_kwargs,
    trust_remote_code=True,
)

print(f"\nModel loaded. GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU: NVIDIA A100-SXM4-40GB — 42.4 GB VRAM
Using Flash Attention 2
GPU: NVIDIA A100-SXM4-40GB — 42.4 GB VRAM
Using Flash Attention 2


Loading checkpoint shards: 100%|██████████| 5/5 [00:04<00:00,  1.10it/s]
Device set to use cuda:0



Model loaded. GPU memory allocated: 16.38 GB

Model loaded. GPU memory allocated: 16.38 GB


## 2. Load LongBench Datasets

In [5]:
from datasets import load_dataset

longbench_datasets = {}
for task_name in LONGBENCH_TASKS:
    ds = load_dataset("THUDM/LongBench", task_name, split="test")
    if FRACTION < 1.0:
        n = max(1, int(len(ds) * FRACTION))
        ds = ds.select(range(n))
    longbench_datasets[task_name] = ds
    print(f"{task_name}: {len(ds)} examples")
    print(f"  Sample input: {ds[0]['input'][:120]}...")
    print(f"  Context length (words): {ds[0]['length']}")
    print(f"  Answers: {ds[0]['answers'][:2]}")
    print()

The repository for THUDM/LongBench contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/THUDM/LongBench.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Generating test split: 200 examples [00:00, 2011.81 examples/s]


gov_report: 2 examples
gov_report: 2 examples
  Sample input: ...
  Context length (words): 8537
  Answers: ["Multiyear procurement (MYP) and block buy contracting (BBC) are special contracting mechanisms that Congress permits the Department of Defense (DOD) to use for a limited number of defense acquisition programs. Compared to the standard or default approach of annual contracting, MYP and BBC have the potential for reducing weapon procurement costs by a few or several percent. Under annual contracting, DOD uses one or more contracts for each year's worth of procurement of a given kind of item. Under MYP, DOD instead uses a single contract for two to five years' worth of procurement of a given kind of item without having to exercise a contract option for each year after the first year. DOD needs congressional approval for each use of MYP. There is a permanent statute governing MYP contracting—10 U.S.C. 2306b. Under this statute, a program must meet several criteria to qualify for MY

Generating test split: 200 examples [00:00, 2329.66 examples/s]

hotpotqa: 2 examples
hotpotqa: 2 examples
  Sample input: Which case was brought to court first Miller v. California or Gates v. Collier ?...
  Sample input: Which case was brought to court first Miller v. California or Gates v. Collier ?...
  Context length (words): 8616
  Answers: ['Miller v. California']
  Context length (words): 8616
  Answers: ['Miller v. California']




## 3. Load Scoring Metrics

In [6]:
import evaluate

rouge_metric = evaluate.load("rouge")
squad_metric = evaluate.load("squad")

TASK_METRICS = {
    "gov_report": rouge_metric,
    "hotpotqa": squad_metric,
}

print("Loaded scoring metrics:")
print("  gov_report → ROUGE-L")
print("  hotpotqa   → SQuAD F1")

Loaded scoring metrics:
  gov_report → ROUGE-L
  hotpotqa   → SQuAD F1
Loaded scoring metrics:
  gov_report → ROUGE-L
  hotpotqa   → SQuAD F1


## 4. Run Inference

For each (algorithm, compression_ratio, LongBench task) combination, run all
examples through the kvpress pipeline. Predictions are collected for scoring
in the next section.

In [7]:
import time

all_results = []

configs = [("no_press", 0.0, None)]
for press_name, press_factory in PRESS_CONFIGS.items():
    for ratio in COMPRESSION_RATIOS:
        configs.append((press_name, ratio, press_factory(ratio)))

for press_name, ratio, press in configs:
    for task_name in LONGBENCH_TASKS:
        ds = longbench_datasets[task_name]
        max_tokens = MAX_NEW_TOKENS[task_name]
        label = f"{press_name} | ratio={ratio} | task={task_name}"
        print(f"\n{'='*60}")
        print(f"Running: {label} ({len(ds)} examples)")
        print(f"{'='*60}")

        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        log_every = max(1, len(ds) // 10)

        for i, row in enumerate(ds):
            kwargs = dict(
                question=row["input"],
                max_new_tokens=max_tokens,
            )
            if press is not None:
                kwargs["press"] = press

            t_start = time.perf_counter()
            output = pipe(row["context"], **kwargs)
            elapsed = time.perf_counter() - t_start

            all_results.append({
                "framework": "kvpress",
                "press": press_name,
                "compression_ratio": ratio,
                "longbench_task": task_name,
                "predicted_answer": output["answer"],
                "reference_answers": row["answers"],
                "elapsed_sec": round(elapsed, 3),
            })

            if (i + 1) % log_every == 0 or (i + 1) == len(ds):
                total_elapsed = time.perf_counter() - t0
                print(f"  {i+1}/{len(ds)} — {total_elapsed:.0f}s elapsed")

        total_elapsed = time.perf_counter() - t0
        peak_mem = torch.cuda.max_memory_allocated() / 1e9
        print(f"  Done: {total_elapsed:.0f}s, peak_mem={peak_mem:.2f}GB")

        torch.cuda.empty_cache()

print(f"\nTotal results: {len(all_results)}")


Running: no_press | ratio=0.0 | task=gov_report (2 examples)

Running: no_press | ratio=0.0 | task=gov_report (2 examples)
  1/2 — 27s elapsed
  1/2 — 27s elapsed
  2/2 — 55s elapsed
  2/2 — 55s elapsed  Done: 55s, peak_mem=20.67GB

  Done: 55s, peak_mem=20.67GB

Running: no_press | ratio=0.0 | task=hotpotqa (2 examples)

Running: no_press | ratio=0.0 | task=hotpotqa (2 examples)
  1/2 — 4s elapsed
  1/2 — 4s elapsed
  2/2 — 9s elapsed
  Done: 9s, peak_mem=20.77GB
  2/2 — 9s elapsed
  Done: 9s, peak_mem=20.77GB

Running: full_replacement | ratio=0.01 | task=gov_report (2 examples)

Running: full_replacement | ratio=0.01 | task=gov_report (2 examples)
  1/2 — 31s elapsed
  1/2 — 31s elapsed
  2/2 — 62s elapsed
  2/2 — 62s elapsed
  Done: 62s, peak_mem=20.66GB
  Done: 62s, peak_mem=20.66GB

Running: full_replacement | ratio=0.01 | task=hotpotqa (2 examples)

Running: full_replacement | ratio=0.01 | task=hotpotqa (2 examples)
  1/2 — 5s elapsed
  1/2 — 5s elapsed
  2/2 — 10s elapsed
  Do

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  2/2 — 62s elapsed
  Done: 62s, peak_mem=20.05GB
  2/2 — 62s elapsed
  Done: 62s, peak_mem=20.05GB

Running: full_replacement | ratio=0.25 | task=hotpotqa (2 examples)

Running: full_replacement | ratio=0.25 | task=hotpotqa (2 examples)
  1/2 — 5s elapsed
  1/2 — 5s elapsed
  2/2 — 10s elapsed
  Done: 10s, peak_mem=20.14GB
  2/2 — 10s elapsed
  Done: 10s, peak_mem=20.14GB

Running: full_replacement | ratio=0.5 | task=gov_report (2 examples)

Running: full_replacement | ratio=0.5 | task=gov_report (2 examples)
  1/2 — 31s elapsed
  1/2 — 31s elapsed
  2/2 — 62s elapsed
  2/2 — 62s elapsed
  Done: 62s, peak_mem=19.43GB
  Done: 62s, peak_mem=19.43GB

Running: full_replacement | ratio=0.5 | task=hotpotqa (2 examples)

Running: full_replacement | ratio=0.5 | task=hotpotqa (2 examples)
  1/2 — 5s elapsed
  1/2 — 5s elapsed
  2/2 — 10s elapsed
  Done: 10s, peak_mem=19.50GB
  2/2 — 10s elapsed
  Done: 10s, peak_mem=19.50GB

Running: full_replacement | ratio=0.75 | task=gov_report (2 examples)

## 5. Score & Results

Score predictions using HuggingFace `evaluate`:
- **gov_report**: ROUGE-L F-measure
- **hotpotqa**: SQuAD token-level F1

In [8]:
import pandas as pd

df = pd.DataFrame(all_results)

all_metrics = {}
rows = []

for (press, ratio, task_name), group in df.groupby(
    ["press", "compression_ratio", "longbench_task"]
):
    preds = group["predicted_answer"].tolist()
    refs = group["reference_answers"].tolist()

    if task_name == "gov_report":
        flat_refs = [r[0] if isinstance(r, list) else r for r in refs]
        result = rouge_metric.compute(predictions=preds, references=flat_refs)
        score = result["rougeL"]
        metric_name = "rougeL"
    else:
        squad_preds = [
            {"prediction_text": p, "id": str(i)}
            for i, p in enumerate(preds)
        ]
        squad_refs = [
            {"answers": {"text": r if isinstance(r, list) else [r],
                         "answer_start": [0] * (len(r) if isinstance(r, list) else 1)},
             "id": str(i)}
            for i, r in enumerate(refs)
        ]
        result = squad_metric.compute(predictions=squad_preds, references=squad_refs)
        score = result["f1"]
        metric_name = "f1"

    key = f"{press}__{ratio}__{task_name}"
    all_metrics[key] = {metric_name: round(score, 4)}
    mean_time = group["elapsed_sec"].mean()
    rows.append({
        "press": press, "compression_ratio": ratio,
        "longbench_task": task_name, "score": round(score, 4),
        "metric": metric_name, "mean_time": round(mean_time, 3),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

           press  compression_ratio longbench_task   score metric  mean_time
       filtering               0.01     gov_report  0.2232 rougeL     91.779
       filtering               0.01       hotpotqa 16.9856     f1     12.679
       filtering               0.25     gov_report  0.2707 rougeL     93.959
       filtering               0.25       hotpotqa 14.5378     f1     12.893
       filtering               0.50     gov_report  0.2386 rougeL     94.124
       filtering               0.50       hotpotqa 14.5614     f1     13.067
       filtering               0.75     gov_report  0.0571 rougeL     93.204
       filtering               0.75       hotpotqa 16.2281     f1     12.903
full_replacement               0.01     gov_report  0.2303 rougeL     30.974
full_replacement               0.01       hotpotqa 16.9856     f1      5.194
full_replacement               0.25     gov_report  0.2335 rougeL     30.912
full_replacement               0.25       hotpotqa 13.0314     f1      5.204

## 6. Save Results

In [9]:
import json

os.makedirs("results/kvpress_longbench", exist_ok=True)

predictions_path = "results/kvpress_longbench/predictions.csv"
df.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/kvpress_longbench/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")

Saved predictions to results/kvpress_longbench/predictions.csv
Saved predictions to results/kvpress_longbench/predictions.csv
Saved metrics to results/kvpress_longbench/metrics.json
Saved metrics to results/kvpress_longbench/metrics.json
